In [1]:
import os
import optuna
import numpy as np
import pandas as pd
import warnings
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
files = [
    'two_class_1s_no.csv',
    'two_class_1s_0.5.csv',
    'two_class_1s_0.8.csv',
    'two_class_2s_no.csv',
    'two_class_2s_0.5.csv',
    'two_class_2s_0.8.csv',
    'two_class_3s_no.csv',
    'two_class_3s_0.5.csv',
    'two_class_3s_0.8.csv',
    'two_class_4s_no.csv',
    'two_class_4s_0.5.csv',
    'two_class_4s_0.8.csv',
    'two_class_5s_no.csv',
    'two_class_5s_0.5.csv',
    'two_class_5s_0.8.csv'
]

base_path = '/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/deployable/data/extracted_features_data/3_class/train_set/'


In [ ]:
class HyperparameterTuner:
    def __init__(self, X, y, groups, n_folds=5, n_trials=50, random_state=42):
        """
        Hyperparameter tuning with GroupKFold CV (no imbalance handling).
        """
        self.X = X
        self.y = LabelEncoder().fit_transform(y)
        self.groups = groups
        self.n_folds = n_folds
        self.n_trials = n_trials
        self.random_state = random_state
        self.cv = StratifiedGroupKFold(n_splits=n_folds)

    def define_search_space(self, trial, model_name):
        """Define search space for each model."""
        if model_name == "rf":
            return {
                "model": "rf",
                "n_estimators": trial.suggest_int("n_estimators", 50, 500),
                "max_depth": trial.suggest_int("max_depth", 3, 20),
                "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
                "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
                "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
            }
        elif model_name == "xgb":
            return {
                "model": "xgb",
                "n_estimators": trial.suggest_int("n_estimators", 50, 500),
                "max_depth": trial.suggest_int("max_depth", 3, 12),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                "subsample": trial.suggest_float("subsample", 0.6, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
                "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
                "gamma": trial.suggest_float("gamma", 0, 5),
                "reg_alpha": trial.suggest_float("reg_alpha", 0, 2),
                "reg_lambda": trial.suggest_float("reg_lambda", 0, 2)
            }
        elif model_name == "dt":
            return {
                "model": "dt",
                "max_depth": trial.suggest_int("max_depth", 1, 20),
                "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
                "criterion": trial.suggest_categorical("criterion", ["gini", "entropy", "log_loss"]),
                "splitter": trial.suggest_categorical("splitter", ["best", "random"]),
            }

    def create_model(self, params):
        """Instantiate model with given parameters."""
        if params["model"] == "rf":
            return RandomForestClassifier(
                n_estimators=params["n_estimators"],
                max_depth=params["max_depth"],
                min_samples_split=params["min_samples_split"],
                min_samples_leaf=params["min_samples_leaf"],
                max_features=params["max_features"],
                bootstrap=params["bootstrap"],
                random_state=self.random_state,
                n_jobs=-1,
            )
        elif params["model"] == "xgb":
            return XGBClassifier(
                n_estimators=params["n_estimators"],
                max_depth=params["max_depth"],
                learning_rate=params["learning_rate"],
                subsample=params["subsample"],
                colsample_bytree=params["colsample_bytree"],
                min_child_weight=params["min_child_weight"],
                gamma=params["gamma"],
                reg_alpha=params["reg_alpha"],
                reg_lambda=params["reg_lambda"],
                random_state=self.random_state,
                n_jobs=-1,
                eval_metric="mlogloss",
                early_stopping_rounds=params["early_stopping_rounds"],
                verbosity=0,
                use_label_encoder=False,
            )
        elif params["model"] == "dt":
            return DecisionTreeClassifier(
                max_depth=params["max_depth"],
                min_samples_split=params["min_samples_split"],
                min_samples_leaf=params["min_samples_leaf"],
                criterion=params["criterion"],
                splitter=params["splitter"],
                random_state=self.random_state,
            )

    def objective(self, trial, model_name):
        """Optuna objective for hyperparameter tuning."""
        params = self.define_search_space(trial, model_name)
        fold_scores = []

        for train_idx, test_idx in self.cv.split(self.X, self.y, self.groups):
            X_train, X_test = self.X.iloc[train_idx], self.X.iloc[test_idx]
            y_train, y_test = self.y[train_idx], self.y[test_idx]

            model = self.create_model(params)

            if params["model"] == "xgb":
                model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
            else:
                model.fit(X_train, y_train)

            y_pred = model.predict(X_test)
            score = accuracy_score(y_test, y_pred)
            fold_scores.append(score)

        return np.mean(fold_scores)

    def tune(self, model_name):
        """Run Optuna optimization for a given model."""
        study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=self.random_state))
        study.optimize(lambda trial: self.objective(trial, model_name), n_trials=self.n_trials, show_progress_bar=True)
        return study.best_params, study.best_value, study

### Tuning

In [4]:
all_results = {}
for file in files:
    data_path = os.path.join(base_path, file)
    features = pd.read_csv(data_path)
    details = file.split('_')
    exp_name = f"{details[2]}_{details[-1].replace('.csv', '')}"
    print(f"Tuning parameters for {exp_name}") 
    
    X = features.drop(columns=['label', 'experiment_id'])
    y = features['label']
    groups = features['experiment_id']
    
    results = {}
    
    for model_name in ['dt', 'rf', 'xgb']:
        print(f"\nTuning {model_name.upper()}...")
        tuner = HyperparameterTuner(X, y, groups, n_folds=5, n_trials=50)
        best_params, best_score, study = tuner.tune(model_name)
        results[model_name] = {'best_params': best_params, 'best_score': best_score, 'study': study}
        print(f"Best {model_name.upper()} Score: {best_score:.4f}")
    
    all_results[f"{exp_name}"] = results
    
    
    

[I 2025-08-21 09:25:42,439] A new study created in memory with name: no-name-1c34522f-ef94-4cb5-8445-aebf0566bc0c


Tuning parameters for 1s_no

Tuning DT...


Best trial: 0. Best value: 0.708105:   4%|▍         | 2/50 [00:00<00:03, 15.04it/s]

[I 2025-08-21 09:25:42,500] Trial 0 finished with value: 0.7081048871421836 and parameters: {'max_depth': 8, 'min_samples_split': 20, 'min_samples_leaf': 8, 'criterion': 'gini', 'splitter': 'random'}. Best is trial 0 with value: 0.7081048871421836.
[I 2025-08-21 09:25:42,572] Trial 1 finished with value: 0.6430368034893699 and parameters: {'max_depth': 13, 'min_samples_split': 15, 'min_samples_leaf': 1, 'criterion': 'gini', 'splitter': 'random'}. Best is trial 0 with value: 0.7081048871421836.
[I 2025-08-21 09:25:42,627] Trial 2 finished with value: 0.6894453996866731 and parameters: {'max_depth': 7, 'min_samples_split': 11, 'min_samples_leaf': 5, 'criterion': 'entropy', 'splitter': 'random'}. Best is trial 0 with value: 0.7081048871421836.


Best trial: 0. Best value: 0.708105:   8%|▊         | 4/50 [00:01<00:13,  3.31it/s]

...

Best trial: 31. Best value: 0.744486:  74%|███████▍  | 37/50 [14:24<02:13, 10.28s/it]

[I 2025-08-21 10:33:43,482] Trial 36 finished with value: 0.7246794367105459 and parameters: {'n_estimators': 52, 'max_depth': 9, 'min_samples_split': 12, 'min_samples_leaf': 6, 'max_features': None, 'bootstrap': True}. Best is trial 31 with value: 0.7444855627525602.


Best trial: 31. Best value: 0.744486:  74%|███████▍  | 37/50 [14:26<05:04, 23.41s/it]


[W 2025-08-21 10:33:45,263] Trial 37 failed with parameters: {'n_estimators': 212, 'max_depth': 12, 'min_samples_split': 16, 'min_samples_leaf': 3, 'max_features': 'sqrt', 'bootstrap': True} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/.venv/lib/python3.12/site-packages/optuna/study/_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_52219/1928231997.py", line 116, in <lambda>
    study.optimize(lambda trial: self.objective(trial, model_name), n_trials=self.n_trials, show_progress_bar=True)
                                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_52219/1928231997.py", line 105, in objective
    model.fit(X_train, y_train)
  File "/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/.venv/lib/python3.12/site-packages/sklearn/base.py", line 1389, in wrapper
 

KeyboardInterrupt: 